In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("crime_summary") \
    .getOrCreate()

df = spark.read.parquet("hdfs://namenode:9000/data/processed/chicago_crimes_clean.parquet")

print("Data loaded:", df.count(), "rows")

Data loaded: 8422096 rows


In [9]:
from pyspark.sql.functions import col, when, count, sum, desc


total_crimes = df.count()

total_arrests = df.filter(col("was_arrest_made") == True).count()
arrest_rate = total_arrests / total_crimes

violent_crimes = df.filter(col("is_violent") == True).count()
violent_crime_rate = violent_crimes / total_crimes

daytime_crimes = df.filter((col("crime_hour") >= 6) & (col("crime_hour") < 18)).count()
nighttime_crimes = total_crimes - daytime_crimes

peak_hour_row = (
    df.groupBy("crime_hour")
      .agg(count("*").alias("crime_count"))
      .orderBy(desc("crime_count"))
      .first()
)
peak_hour = peak_hour_row["crime_hour"]

peak_crime_type_row = (
    df.groupBy("Primary Type")
      .agg(count("*").alias("crime_count"))
      .orderBy(desc("crime_count"))
      .first()
)
peak_crime_type = peak_crime_type_row["Primary Type"]

peak_district_row = (
    df.groupBy("District")
      .agg(count("*").alias("crime_count"))
      .orderBy(desc("crime_count"))
      .first()
)
peak_district = peak_district_row["District"]

violent_by_dist = (
    df.groupBy("District")
      .agg(
          count("*").alias("total"),
          sum(when(col("is_violent") == True, 1).otherwise(0)).alias("violent_count")
      )
      .withColumn("violent_rate", col("violent_count") / col("total"))
      .orderBy(desc("violent_rate"))
      .first()
)

most_violent_district = violent_by_dist["District"]

In [10]:
summary = {
    "total_crimes": total_crimes,
    "total_arrests": total_arrests,
    "arrest_rate": arrest_rate,
    "violent_crime_rate": violent_crime_rate,
    "daytime_crimes": daytime_crimes,
    "nighttime_crimes": nighttime_crimes,
    "peak_hour": peak_hour,
    "peak_crime_type": peak_crime_type,
    "peak_district": peak_district,
    "most_violent_district": most_violent_district
}

print(summary)

{'total_crimes': 8422096, 'total_arrests': 2124978, 'arrest_rate': 0.2523098763063256, 'violent_crime_rate': 0.30629857460660626, 'daytime_crimes': 4322671, 'nighttime_crimes': 4099425, 'peak_hour': 0, 'peak_crime_type': 'THEFT', 'peak_district': 8, 'most_violent_district': 21}


In [11]:
spark.stop()